### Imports & Paths

In [1]:
# Complete Unified Inference Pipeline for Deliverable 3

import os
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.special import softmax

import torch
import torch.nn as nn

from sentence_transformers import SentenceTransformer
import faiss

from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from captum.attr import LayerIntegratedGradients

import re

### Paths & Device Setup

In [2]:
# Base directory
BASE = Path("/Users/satwik/Documents/GitHub/UF-EEE6778-Fall25-TermProject")

# Retrieval files
FAISS_INDEX_FILE = BASE / "data/processed/faiss_index/claimverify_faiss_index.bin"
FAISS_META_FILE  = BASE / "data/processed/faiss_index/claimverify_faiss_metadata.csv"

# Classifier v2
MODEL_DIR = BASE / "models/classifier/roberta_finetuned_v2"

# Load temperature file
TEMP_FILE = MODEL_DIR / "temperature_scaling.pt"

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

device

device(type='mps')

### Preprocessing

In [3]:
def normalize_text(text):
    """
    Deliverable 3 text normalization.
    Matches classifier-preprocessing rules.
    """
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s.,?\"'-]", "", text)
    return text


### Load FAISS Index + Metadata + Embedding Model

In [4]:
# Load metadata
print("📂 Loading metadata...")
metadata = pd.read_csv(FAISS_META_FILE)

# Load FAISS index
print("📂 Loading FAISS index...")
index = faiss.read_index(str(FAISS_INDEX_FILE))
print(f"   Loaded FAISS with {index.ntotal} vectors.")

# Load SentenceTransformer model for retrieval
print("⚙️ Loading SentenceTransformer model...")
retrieval_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("   Retrieval model loaded.")

📂 Loading metadata...
📂 Loading FAISS index...
   Loaded FAISS with 25540 vectors.
⚙️ Loading SentenceTransformer model...
   Retrieval model loaded.


### Load Improved Classifier + Tokenizer + Temp Scaling

In [5]:
# Load tokenizer
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_DIR)

# Load model
model = RobertaForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()

# Load label map
import pickle
with open(MODEL_DIR / "label_mapping.pkl", "rb") as f:
    label_map = pickle.load(f)
id2label = {v: k for k, v in label_map.items()}

# Load temperature
temp_val = torch.load(TEMP_FILE)["temperature"]
temperature = temp_val
temperature

1.1678239107131958

### Function: FAISS Retrieval

In [6]:
def retrieve_offline_evidence(query, top_k=5):
    """
    Offline FAISS retrieval with MiniLM embeddings.
    """
    start = time.time()

    # Normalize & embed
    clean_query = normalize_text(query)
    q_embed = retrieval_model.encode(clean_query, normalize_embeddings=True)
    q_embed = np.array([q_embed]).astype("float32")

    # FAISS search
    scores, idxs = index.search(q_embed, top_k)

    elapsed = (time.time() - start) * 1000

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], idxs[0])):
        row = metadata.iloc[idx]
        results.append({
            "rank": rank + 1,
            "claim_id": row["claim_id"],
            "claim_text": row["claim_text"],
            "similarity": float(score),
            "verdict_mapped": row["verdict_mapped"],
            "summary": row["summary"],
            "url": row["url"],
            "dataset_source": row["dataset_source"]
        })

    return results, elapsed


### Google Fallback (stub)

In [7]:
# Responding to Deliverable-3 requirement; real API can be implemented in D4)

def google_fallback_stub(query):
    """
    Google API fallback stub for Deliverable 3.
    Returns mock results if similarity < threshold.
    """
    return [
        {
            "rank": 1,
            "claim_id": "G00001",
            "claim_text": f"Google fallback result for: {query}",
            "similarity": 0.42,
            "verdict_mapped": "Unknown",
            "summary": "Could not match claim in offline DB.",
            "url": "https://google.com/search?q=fallback",
            "dataset_source": "GoogleFallbackStub"
        }
    ]

### Function: Classifier Prediction (with temperature scaling)

In [8]:
def classify_claim_text(text):
    """
    Deliverable 3 classifier inference:
    - normalized text
    - calibrated logits
    """
    start = time.time()

    text_clean = normalize_text(text)
    enc = tokenizer(
        text_clean,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )

    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits
        logits = logits / temperature
        probs = softmax(logits.cpu().numpy(), axis=1)[0]

    pred_idx = int(np.argmax(probs))
    pred_label = id2label[pred_idx]
    pred_conf = float(np.max(probs))

    elapsed = (time.time() - start) * 1000

    return pred_label, pred_conf, probs, elapsed


### Integrated Gradients Explainability

In [12]:
# Captum LIG setup
embedding_layer = model.roberta.embeddings.word_embeddings
lig = LayerIntegratedGradients(lambda ids, mask: model(ids, mask).logits, embedding_layer)

def explain_with_ig(text, target_idx):
    target_idx = int(target_idx)
    start = time.time()

    enc = tokenizer(
        normalize_text(text),
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    input_ids = enc["input_ids"].to(device)
    attn = enc["attention_mask"].to(device)

    baseline = torch.zeros_like(input_ids).to(device)

    atts, _ = lig.attribute(
        input_ids,
        baselines=baseline,
        additional_forward_args=(attn,),
        target=int(target_idx), 
        n_steps=50,
        return_convergence_delta=True
    )

    token_atts = atts.sum(dim=-1).squeeze().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze().cpu())

    results = []
    for tok, score in zip(tokens, token_atts):
        if tok not in ["<s>", "</s>", "<pad>"]:
            results.append({"token": tok, "score": float(score)})

    elapsed = (time.time() - start) * 1000
    return results, elapsed


### Unified Inference Function (Deliverable 3)

In [13]:
def claimverify_infer(user_claim):
    overall_start = time.time()


    # 1) Retrieval
    evidence, t_retrieval = retrieve_offline_evidence(user_claim)

    top1_sim = evidence[0]["similarity"]


    # 2) Google fallback
    if top1_sim < 0.70:
        fallback = google_fallback_stub(user_claim)
        verdict = "Low Coverage"
        return {
            "verdict": verdict,
            "confidence": None,
            "evidence": fallback,
            "explanation": None,
            "source": "Hybrid Fallback",
            "runtime_ms": {
                "retrieval": t_retrieval,
                "classification": None,
                "explainability": None,
                "total": (time.time() - overall_start) * 1000
            }
        }

    # 3) Classifier prediction
    pred_label, pred_conf, probs, t_class = classify_claim_text(user_claim)


    # 4) Uncertainty thresholding
    if pred_conf < 0.55:
        pred_label = "Uncertain"

    # 5) Explanation
    target_idx = int(label_map[pred_label])  # Pure Python int
    explanation, t_expl = explain_with_ig(user_claim, target_idx)

    total_time = (time.time() - overall_start) * 1000

    # -------------------------
    # 6) Final structured output
    # -------------------------
    return {
        "verdict": pred_label,
        "confidence": float(pred_conf),
        "evidence": evidence,
        "explanation": explanation,
        "source": "OfflineDB",
        "runtime_ms": {
            "retrieval": t_retrieval,
            "classification": t_class,
            "explainability": t_expl,
            "total": total_time
        }
    }


In [14]:
### Testing the Complete System

query = "COVID-19 vaccines cause infertility."

result = claimverify_infer(query)
result

{'verdict': 'Likely False',
 'confidence': 0.9772477746009827,
 'evidence': [{'rank': 1,
   'claim_id': 'P12748',
   'claim_text': 'University of Miami researchers have found that the COVID-19 vaccine affects sperm production.',
   'similarity': 0.7758840322494507,
   'verdict_mapped': 'Likely False',
   'summary': nan,
   'url': 'https://www.politifact.com/factchecks/2021/may/19/viral-image/university-miami-researchers-looked-effects-covid-/',
   'dataset_source': 'PolitiFact'},
  {'rank': 2,
   'claim_id': 'P20071',
   'claim_text': 'Women’s menstrual cycles and fertility are affected by being around people who have received COVID-19 vaccines',
   'similarity': 0.7671603560447693,
   'verdict_mapped': 'Likely False',
   'summary': nan,
   'url': 'https://www.politifact.com/factchecks/2021/apr/21/facebook-posts/no-womens-cycles-and-fertility-are-not-affected-be/',
   'dataset_source': 'PolitiFact'},
  {'rank': 3,
   'claim_id': 'P00936',
   'claim_text': 'The COVID-19 vaccines cause A